# Masknmf: ASAP7 voltage indicator, single channel

In [ ]:
from pathlib import Path

import fastplotlib as fpl
import matplotlib.pyplot as plt
import numpy as np

import mbo_utilities as mbo
import masknmf

In [ ]:
MESC_PATH = Path("D:/demo/scan.mesc").expanduser()
UNIT = "MSession_0/MUnit_0"
CHANNEL = 0
EXCLUDE_INITIAL_FRAMES = 200
OUTPUT_FOLDER = MESC_PATH.parent / "masknmf_asap7" / UNIT.replace("/", "_")
DEVICE = "cuda"

In [ ]:
MESC_PATH.is_file()

## Load and invert indicator

In [ ]:
arr = mbo.imread(MESC_PATH, unit=UNIT)
fs = arr.fs

raw = arr[EXCLUDE_INITIAL_FRAMES:, CHANNEL, 0]
movie = masknmf.OphysArray(raw, negative_indicator=True, include_mean=True) #, device="cpu") # [:].numpy()
movie_nomean = masknmf.OphysArray(raw, negative_indicator=True, include_mean=False) #, device="cpu") # [:].numpy()
movie.shape

In [ ]:
movie = masknmf.OphysArray(raw, negative_indicator=True)

panels = {"raw": raw, "inverted": movie}
ndw = fpl.NDWidget(names=list(panels), shape=(1, 2), size=(1200, 400))
for name, data in panels.items():
  image = ndw[name].add_nd_image(data, ["time", "m", "n"], ["m", "n"], spatial_func=None, name=name)
  image.graphic.cmap = "gray"
ndw.show()

## Denoise before registration

tiny movie, tiny block size

In [ ]:
block_sizes = [4, 4]
moco_strategy = masknmf.CompressStrategy(block_sizes=block_sizes,
                                         max_components=20,
                                         max_consecutive_failures=1,
                                         temporal_avg_factor=2,
                                         spatial_avg_factor=4,
                                         frame_batch_size=300)
pmd_moco = moco_strategy.compress(movie)
pmd_moco.to(DEVICE)

## Register

Shifts are estimated on the denoised movie and applied to the raw frames.

In [ ]:
corrector = masknmf.RigidMotionCorrector(max_shifts=[40, 40])
corrector.compute_template(pmd_moco)
moco_array = corrector.motion_correct(reference_movie=pmd_moco, target_movie=movie)

registered = moco_array[:].cpu().numpy()
pmd_registered = corrector.motion_correct(reference_movie=pmd_moco)[:].cpu().numpy()
shifts = moco_array.shifts.cpu().numpy()

## Compare

In [ ]:
panels = {
    "raw": movie,
    "pmd(raw)": pmd_moco,
    "registered": registered,
    "pmd(registered)": pmd_registered,
}
ndw = fpl.NDWidget(names=list(panels), shape=(2, 2), size=(900, 700))
for name, data in panels.items():
    image = ndw[name].add_nd_image(data, ["time", "m", "n"], ["m", "n"], spatial_func=None, name=name)
    image.graphic.cmap = "gray"
ndw.show()

In [ ]:
from masknmf.demixing.demixing_results import DemixingResults
from masknmf.visualization.demixing_vis import SingleSessionDemixingVis

In [ ]:
vis = SingleSessionDemixingVis(moco_array, raw=movie)
vis.show()

The same registration with no denoising: template and shifts estimated on the raw movie.

In [ ]:
corrector_raw = masknmf.RigidMotionCorrector(max_shifts=[40, 40])
corrector_raw.compute_template(movie)
shifts_raw = corrector_raw.motion_correct(movie).shifts.cpu().numpy()

In [ ]:
t = (EXCLUDE_INITIAL_FRAMES + np.arange(movie.shape[0])) / fs
figure = fpl.Figure(shape=(2, 1), names=[["y shift (px)"], ["x shift (px)"]], size=(1000, 500))
for subplot, k in zip(figure, (0, 1)):
  subplot.add_line(np.column_stack([t, shifts_raw[:, k]]).astype(np.float32), colors="gray", name="estimated on raw")
  subplot.add_line(np.column_stack([t, shifts[:, k]]).astype(np.float32), colors="royalblue", name="estimated on denoised")
figure[0, 0].controller.add_camera(figure[1, 0].camera, include_state={"x", "width"})
figure[1, 0].controller.add_camera(figure[0, 0].camera, include_state={"x", "width"})
figure.show(maintain_aspect=False)


## Pipeline


In [ ]:
# `mad_correlation_threshold` decreased to  0.6 instead of 0.8 where no signals are found
MAD_CORRELATION_THRESHOLD = 0.6

passes = []
for _ in range(2):
    init_config = masknmf.SuperpixelInitConfig(mad_correlation_threshold=MAD_CORRELATION_THRESHOLD,
                                               residual_threshold=0.1,
                                               sign="positive")
    nmf_config = masknmf.NMFConfig(support_threshold=(0.95, 0.7),
                                   ring_model_start_pt=41,
                                   min_brightness=0.0,
                                   merge_threshold=0.7,
                                   reassign_background=False)
    passes.append(masknmf.SinglepassDemixingConfig(init_config, nmf_config))
demixing_config = masknmf.MultipassDemixingConfig(passes)

In [ ]:
pipeline = masknmf.GlutamateCalciumSpinePipeline(output_folder=OUTPUT_FOLDER,
                                                 demixing_config=demixing_config,
                                                 device=DEVICE)
run_folder = pipeline.run(glutamate_channel=movie,
                          calcium_channel=None,
                          exclude_initial_frames=0)
run_folder

## Results

To reopen an earlier run, set `run_folder` to it and skip the run cell.

In [ ]:
results_path = run_folder / "results.glutamate.hdf5"
spines = masknmf.DemixingResults.from_hdf5(results_path, device=DEVICE)
dendrite = masknmf.DemixingResults.from_hdf5(results_path, prefix="global", device=DEVICE)
spines.spatial_demixed.shape, dendrite.spatial_demixed.shape

In [ ]:
vis = masknmf.SingleSessionDemixingVis(spines,
                                       frame_timings=t,
                                       results_path=results_path,
                                       raw=movie,
                                       device=DEVICE)
vis.show()